In [ ]:
%pip install -q langchain langchain-groq langchain-community sentence-transformers chromadb pandas openpyxl python-dotenv langchain-huggingface langchain-chroma
print("Installation complete. You may need to restart the kernel if this is the first time running.")

Installation complete. You may need to restart the kernel if this is the first time running.


In [ ]:
# Noah Russell Assignment 2 Question 7
import os
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# --- ENTER YOUR KEYS HERE ---
os.environ["GROQ_API_KEY"] = ""
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

print("Keys set successfully.")





Keys set successfully.


In [ ]:
# 1. Load the Excel file
file_path = "feedback.xlsx"  # Make sure your file is named this!

if os.path.exists(file_path):
    df = pd.read_excel(file_path)
    print(f"Data loaded. Found {len(df)} rows.")
else:
    print(f"ERROR: '{file_path}' not found. Please upload the file.")

# 2. Convert rows to Documents
documents = []
for index, row in df.iterrows():
    # Convert row to string
    content = "\n".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])
    doc = Document(page_content=content, metadata={"row": index})
    documents.append(doc)

# 3. Create Vector Database (Embeddings)
print("Creating Vector Store... (This might take a minute)")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)

print("Vector Database created successfully!")

Data loaded. Found 134 rows.
Creating Vector Store... (This might take a minute)
Vector Database created successfully!


In [ ]:
# 1. Define the Retrieval Tool
@tool
def get_feedback(query: str) -> str:
    """Searches student feedback for similar answers."""
    # Note: We use the vectorstore defined in the previous cell
    docs = vectorstore.similarity_search(query, k=5)
    return "\n\n".join([d.page_content for d in docs])

# 2. Initialize the LLM (Groq)
# Using a currently supported model
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

# 3. Create the Agent
tools = [get_feedback]
# We use a simple prompt to guide the agent
agent = create_agent(llm, tools, system_prompt="You are a helpful assistant analyzing student feedback.")

print("Agent is ready to answer questions! (Model updated)")

Agent is ready to answer questions! (Model updated)


In [ ]:
# Query
query_7a = "machine learning and vision"  # This can be changed if necessary

# Get Top 2 Exact Matches from Vector Store
print(f"--- Q7a: Top 2 Answers for '{query_7a}' ---")
docs = vectorstore.similarity_search(query_7a, k=2)

for i, doc in enumerate(docs):
    print(f"\n[Answer {i+1}]:")
    print(doc.page_content)

print("\n--- Explanation ---")
print("The LLM/Vector Store is accurate because it uses semantic embeddings (HuggingFace) to find conceptual matches, not just keyword matches.")

--- Q7a: Top 2 Answers for 'machine learning and vision' ---

[Answer 1]:
z_last_digits: 571220
student_writeup: I hope to acquire solid theoretical and practical computer vision skills, such as segmentation, object recognition, feature detection, and image generation. I'm interested in learning about the design, training, and evaluation of both traditional vision techniques and contemporary deep learning methods (such CNNs). Understanding constraints and biases is crucial, as is developing intuition for selecting suitable models, loss functions, and datasets. Additionally, I want to gain more practical experience with reading research articles, applying computer vision approaches to real-world issues, and constructing vision algorithms.

[Answer 2]:
z_last_digits: 21522
student_writeup: I would like to learn how to apply artificial intelligence to my research with latency in object detection. I want to expand on current learning models and determine how to achieve the lowest latency w

In [ ]:
# Ask the Agent to summarize
query_7b = "Summarize the main topics students are interested in. List the top 5 most common topics."

print("--- Q7b: Agent Summary ---")
response = agent.invoke({"messages": [{"role": "user", "content": query_7b}]})
print(response['messages'][-1].content)

--- Q7b: Agent Summary ---
Based on the student write-ups, the top 5 most common topics that students are interested in learning from this class on Artificial Intelligence are:

1. **Problem-solving strategies**: Students want to learn how to apply problem-solving strategies in AI agents, including search methods, knowledge representation, and reasoning.
2. **Machine learning and neural networks**: Students are interested in learning about machine learning and neural networks, including how to create, train, and assess AI models.
3. **AI applications in real-world scenarios**: Students want to learn how to apply AI concepts to real-world problems and scenarios, including data analysis, AI technologies, and results communication.
4. **Reinforcement learning**: Students are interested in learning about reinforcement learning and how systems can learn from examples.
5. **AI ethics and limitations**: Students want to understand the ethical challenges and limitations of AI, including its co